# Getting Started with TSPM-DB

This notebook walks through the complete workflow for setting up a TSPM-DB database, ingesting a real-world COVID-19 EHR dataset, and calculating transitive sequences.

**Dataset:** COVID_35k_subset — a 35,000-patient synthetic COVID-19 EHR dataset.

**What you will learn:**
1. How to open or create a TSPM-DB database
2. How to ingest CSV data from a ZIP archive
3. How to calculate transitive sequences and frequencies
4. How to verify the results

## 1. Setup

Make sure the `src/` directory is on your Python path, then import the library.

In [ ]:
import sys
sys.path.insert(0, '../src')

import tspmdb
print('tspmdb imported successfully')

## 2. Open the Database

Create a new TSPM-DB database file. We set `destructive=True` so that re-running this notebook starts fresh.

- `parallel_threads`: number of CPU cores to use during sequence calculation
- `max_memory_mb`: total memory budget across all worker processes

In [ ]:
DB_PATH  = '../covid_35k.sqlite3'
ZIP_FILE = '../data/COVID_35k_subset.zip'
CSV_FILE = 'COVID_35k_subset.csv'

db = tspmdb.TspmDB(
    DB_PATH,
    destructive=True,
    parallel_threads=4,
    max_memory_mb=4096
)

print('Database opened:', DB_PATH)

## 3. Ingest the Dataset

The `col_names` dictionary maps TSPM's four required field names to the actual column names in the CSV file.

| TSPM field | CSV column | Description |
|---|---|---|
| `PATIENT` | `patient_id` | Unique patient identifier |
| `DATE` | `obs_date` | Date of the observation |
| `CODE` | `obs_code` | Observation code (ICD-10, RxNorm, etc.) |
| `TEXT` | `obs_description` | Human-readable description |

In [ ]:
import time

col_names = {
    'PATIENT': 'patient_id',
    'DATE':    'obs_date',
    'CODE':    'obs_code',
    'TEXT':    'obs_description'
}

t0 = time.perf_counter()
db.dataset.ingest(CSV_FILE, col_names, zip_file=ZIP_FILE, show_progress=True)
t1 = time.perf_counter()

print(f'Ingest completed in {t1 - t0:.1f} seconds')

### Verify the ingested data

In [ ]:
cur = db.conn.cursor()

cur.execute('SELECT COUNT(*) FROM lookup_patients')
n_patients = cur.fetchone()[0]

cur.execute('SELECT COUNT(*) FROM lookup_observations')
n_obs_codes = cur.fetchone()[0]

cur.execute('SELECT COUNT(*) FROM source_data')
n_records = cur.fetchone()[0]

print(f'Patients:          {n_patients:,}')
print(f'Unique obs codes:  {n_obs_codes:,}')
print(f'Total records:     {n_records:,}')

## 4. Calculate Sequences and Frequencies

This is the core TSPM computation step. It:
1. Generates all transitive sequence pairs per patient
2. Aggregates population-level frequencies
3. Applies a sparsity filter (default: keep sequences seen in ≥5% of patients)
4. Stores the filtered sequences and frequencies in the database

We optionally define **temporal buckets** to group the raw day-count distances into meaningful clinical ranges.

In [ ]:
temporal_buckets = [
    (0,   1),    # same day or next day
    (1,   7),    # within a week
    (7,   30),   # within a month
    (30,  365),  # within a year
]

t0 = time.perf_counter()
db.dataset.calculate(
    temporal_buckets=temporal_buckets,
    sparsity_threshold=0.05
)
t1 = time.perf_counter()

print(f'Calculation completed in {t1 - t0:.1f} seconds')

### Verify the calculated results

In [ ]:
cur.execute('SELECT COUNT(*) FROM sequences')
n_sequences = cur.fetchone()[0]

cur.execute('SELECT COUNT(*) FROM frequencies')
n_frequencies = cur.fetchone()[0]

print(f'Sequence records:   {n_sequences:,}')
print(f'Frequency records:  {n_frequencies:,}')

## 5. Quick Preview of Results

Let's take a quick look at the top frequency records — the most common observation sequences across the population.

In [ ]:
import pandas as pd

freq_df = db.population.frequencies(as_pandas=True)
print(f'Total frequency rows: {len(freq_df):,}')
freq_df.sort_values('patient_cnt', ascending=False).head(10)

## 6. Close the Database

Always close the database when you are done.

In [ ]:
db.close()
print('Database closed.')

## Next Steps

- **[02_exploring_population.ipynb](02_exploring_population.ipynb)** — Query patients, sequences, and frequencies across the full population.
- **[03_subpopulations.ipynb](03_subpopulations.ipynb)** — Define patient cohorts and compare their sequence profiles.